# 13 — Produtos TOTVS e termos principais

Este módulo transforma a transcrição em candidatos fundamentados na base TOTVS e em até dez termos úteis para a revisão comercial. O score do ranking é heurístico e relativo, não uma probabilidade de compra.

## Representação lexical

Campos mais específicos — produto, título e palavras-chave — recebem mais peso. Os aliases reconhecem como equivalentes formas como `Protheus` e `TOTVS Protheus`.

In [ ]:
def _document_tokens(document: dict[str, Any]) -> list[str]:
    weighted_fields = [
        (document.get("product", ""), 4),
        (document.get("title", ""), 3),
        (" ".join(document.get("keywords", [])), 3),
        (" ".join(document.get("competitors", [])), 2),
        (document.get("category", ""), 1),
        (" ".join(document.get("segments", [])), 1),
        (" ".join(document.get("related_products", [])), 1),
        (document.get("content", ""), 1),
    ]
    return [token for text, weight in weighted_fields for token in _tokens(text) * weight]

def _query_tokens(transcription: str, alias_groups: list[dict[str, Any]]) -> tuple[list[str], set[str]]:
    normalized = f" {_normalize(transcription)} "
    query = _tokens(transcription)
    explicit_products: set[str] = set()
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        if any(f" {_normalize(variant)} " in normalized for variant in variants):
            canonical = group["canonical"]
            explicit_products.add(_normalize(canonical))
            query.extend(_tokens(canonical) * 3)
    return query, explicit_products


## Ranking com fontes

O cálculo BM25 consolida documentos pelo produto, exige ao menos uma URL de fonte e conserva no máximo três candidatos. `explicit_match` distingue uma citação direta de uma correspondência apenas contextual.

In [ ]:
def _rank_products(transcription: str, top_k: int = 3) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    if not query:
        return []

    document_tokens = [_document_tokens(document) for document in knowledge_base]
    frequencies = [Counter(tokens) for tokens in document_tokens]
    lengths = [len(tokens) for tokens in document_tokens]
    average_length = sum(lengths) / max(len(lengths), 1)
    document_frequency = Counter()
    for tokens in document_tokens:
        document_frequency.update(set(tokens))

    grouped: dict[str, dict[str, Any]] = {}
    for index, (document, tokens, term_frequency, length) in enumerate(
        zip(knowledge_base, document_tokens, frequencies, lengths)
    ):
        score = 0.0
        for term in query:
            frequency = term_frequency.get(term, 0)
            if not frequency:
                continue
            seen_in = document_frequency[term]
            inverse = math.log(1 + (len(knowledge_base) - seen_in + 0.5) / (seen_in + 0.5))
            denominator = frequency + 1.5 * (1 - 0.75 + 0.75 * length / average_length)
            score += inverse * frequency * 2.5 / denominator

        product = document.get("product") or document.get("title")
        normalized_product = _normalize(product)
        explicit_match = any(
            alias in normalized_product
            or normalized_product in alias
            or set(_tokens(alias)).issubset(set(_tokens(product)))
            for alias in explicit_products
        )
        if explicit_match:
            score += 8.0
        if score < 2.0:
            continue

        source_urls = [source["url"] for source in document.get("sources", []) if source.get("url")]
        if not source_urls:
            continue
        matched = set(query) & set(tokens)
        candidate = grouped.setdefault(
            product,
            {
                "raw_score": 0.0,
                "explicit_match": False,
                "matched_terms": set(),
                "document_ids": [],
                "sources": [],
            },
        )
        candidate["raw_score"] = max(candidate["raw_score"], score)
        candidate["explicit_match"] = candidate["explicit_match"] or explicit_match
        candidate["matched_terms"].update(matched)
        candidate["document_ids"].append(document["id"])
        candidate["sources"].extend(source_urls)

    ranked = sorted(grouped.items(), key=lambda item: (-item[1]["raw_score"], item[0]))[:top_k]
    if not ranked:
        return []
    highest_score = ranked[0][1]["raw_score"]
    return [
        {
            "product": product,
            "score": round(values["raw_score"] / highest_score, 6),
            "score_type": "heuristic",
            "engine": "bm25_aliases",
            "model": None,
            "explicit_match": values["explicit_match"],
            "matched_terms": sorted(values["matched_terms"])[:10],
            "document_ids": list(dict.fromkeys(values["document_ids"])),
            "sources": list(dict.fromkeys(values["sources"])),
        }
        for product, values in ranked
    ]


## Proteção durante a extração de termos

E-mail, telefone e CPF são removidos somente da cópia usada para extrair termos. Essa filtragem não altera `transcricao_original`.

In [ ]:
def _text_without_personal_data(text: str) -> str:
    sanitized = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " ", text, flags=re.I)
    sanitized = re.sub(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b", " ", sanitized)
    sanitized = re.sub(
        r"(?<!\d)(?:\+?55\s*)?(?:\(?\d{2}\)?[\s.-]*)?9?\d{4}[\s.-]*\d{4}(?!\d)",
        " ",
        sanitized,
    )
    return sanitized


## Priorização dos termos

Produtos citados vêm primeiro, seguidos de dores, concorrentes e vocabulário comercial. A frequência geral completa a lista somente quando ainda há espaço, sempre com limite de dez itens únicos.

In [ ]:
def _extract_key_terms(transcription: str, limit: int = 10) -> list[str]:
    sanitized = _text_without_personal_data(transcription)
    normalized = _normalize(sanitized)
    knowledge_base, alias_groups = _load_catalog()
    prioritized: list[str] = []
    seen: set[str] = set()

    def add(term: str) -> None:
        canonical = _normalize(term).strip()
        if not canonical or canonical in seen or len(prioritized) >= limit:
            return
        prioritized.append(term)
        seen.add(canonical)

    product_terms: list[tuple[int, str]] = []
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        positions = [
            normalized.find(_normalize(variant))
            for variant in variants
            if re.search(rf"(?<!\w){re.escape(_normalize(variant))}(?!\w)", normalized)
        ]
        if positions:
            product_terms.append((min(positions), group["canonical"]))
    for _, term in sorted(product_terms):
        add(term)

    pain_hits = _signal_hits(normalized, OPPORTUNITY_PAIN_SIGNALS)
    for term in sorted(pain_hits, key=lambda item: (normalized.find(item), item)):
        add(term)

    competitors = {
        competitor
        for document in knowledge_base
        for competitor in document.get("competitors", [])
        if competitor
    }
    competitor_hits = [
        competitor
        for competitor in competitors
        if re.search(
            rf"(?<!\w){re.escape(_normalize(competitor))}(?!\w)", normalized
        )
    ]
    for term in sorted(competitor_hits, key=lambda item: (normalized.find(_normalize(item)), item)):
        add(term)

    commercial_signals = (
        OPPORTUNITY_INTENT_SIGNALS
        | OPPORTUNITY_BUY_SIGNALS
        | set(HIGH_CHURN_SIGNALS)
        | set(MEDIUM_CHURN_SIGNALS)
    )
    commercial_hits = _signal_hits(normalized, commercial_signals)
    for term in sorted(commercial_hits, key=lambda item: (normalized.find(item), item)):
        add(term)

    token_counts = Counter(_tokens(sanitized))
    first_position = {token: normalized.find(token) for token in token_counts}
    for token, _ in sorted(
        token_counts.items(),
        key=lambda item: (-item[1], first_position[item[0]], item[0]),
    ):
        if token.isdigit() or any(
            re.search(rf"(?<!\w){re.escape(token)}(?!\w)", existing)
            for existing in seen
        ):
            continue
        add(token)
    return prioritized
